## RSNA knee: CNN-MIL training (ConvNeXt-T / CoAtNet-2), Colab, one clean pipeline
Everything an experiment can change lives in the **RUN SETTINGS** cell below; nothing else needs editing.
The core cells handle every option directly (no patch cells), and the run name is derived from the settings,
so two different set-ups can never share a run folder.

| Setting | Options |
|---|---|
| `layout` | `"ontology5"` (22 triplets) · `"dense5"` (all cached slices, 40 triplets) |
| `overlap_stride` | `None` · `2` (55 triplets) · `1` (110 triplets); dense5 only |
| `laterality` | `True` = every knee shown as a right knee |
| `oai` | `None` · `"A"` (PF OA / Lateral OA / Synovitis) · `"B"` (9 MOAKS labels), with `oai_weight` |
| `img`, `folds`, `patience`, `epochs`, `save_every_epoch`, `select_label`, `tag` | as named |

Run: A100 runtime → *Run all*. The check cell prints the full set-up and stops on anything inconsistent.
Folds already finished under the same name are skipped, so a disconnect resumes where it stopped.
Data rules: OAI images, cache and labels stay on your Drive (OAI data use agreement); only trained weights are uploaded.


In [ ]:
# ============================ RUN SETTINGS: the only cell to edit ============================
RUN = {
    "arch": "convnext_tiny",        # "convnext_tiny" | "coatnet2"
    "labels": "a50new",             # training labels: MyDrive/rsna_labels/train_labels_<labels>.csv
    "val_labels": "a50new",         # labels the per-epoch validation AUC is computed on
    "layout": "dense5",             # "ontology5" | "dense5"
    "overlap_stride": None,         # None | 2 | 1   (dense5 only)
    "laterality": True,
    "oai": None,                    # None | "A" | "B"
    "oai_weight": 0.5,              # loss weight of an OAI label relative to an RSNA label
    "img": None,                    # None = the arch default (256 ConvNeXt, 224 CoAtNet)
    "folds": [0, 1, 2, 3, 4],       # finished folds are skipped
    "patience": 3,                  # None = full schedule
    "epochs": 16,
    "save_every_epoch": False,      # per-epoch OOF csvs (screens)
    "select_label": None,           # None = checkpoint on macro AUC
    "tag": "",                      # optional suffix for a one-off variant
    "smoke": False,                 # True = 200 train / 100 valid studies, 1 epoch, nothing marked done
}
# Examples. Fold-3 screen:      RUN.update(folds=[3], patience=None, save_every_epoch=True)
#           Stride-2 5 folds:   RUN.update(layout="dense5", overlap_stride=2)
#           dense5 + OAI A 0.5: RUN.update(layout="dense5", oai="A", oai_weight=0.5)
#           384 px:             RUN.update(img=384)

# Final general models (the leg in the submission): a50v2 labels, dense5 + stride-2 windows, mirrored left knees.
# Seeds 42, 1, 2 and 3 were run with this setting (CFG["seed"] in the Config cell); see README "Training".
RUN.update(arch="convnext_tiny", labels="a50v2", val_labels="a50new", layout="dense5", overlap_stride=2,
           laterality=True, oai=None, folds=[0, 1, 2, 3, 4], patience=3)

In [ ]:
# ======================= Environment (Colab): Drive, Kaggle inputs, cache shards =======================
import os, sys, json, shutil, subprocess, time
from pathlib import Path
import pandas as pd

assert "google.colab" in sys.modules, "this notebook is the Colab version"
from google.colab import userdata, drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount("/content/drive", force_remount=True)

SETUP = {
    "cache_key": "0266fa68",                                   # tissue-centred cache
    "drive_shards": "/content/drive/MyDrive/rsna_shards",      # series_cache_0266fa68_shard{0..5}.tar
    "drive_labels": "/content/drive/MyDrive/rsna_labels",      # training label CSVs
    "label_files": ["train_processed_v3_1.csv", "folds_v31_scanner.csv", "dicom_study_metadata_train.csv"],
    "label_handles": ["harrisonhoran/rsna-knee-translatedprocessed-train-files",
                      "harrisonhoran/harry-rsna-dicom-metadata-dataset"],
    "competition": "rsna-knee-abnormality-detection",
    "data_dir": "/content/data",                               # local disk; never train from the Drive mount
    "runs_dir": "/content/drive/MyDrive/rsna_runs",
    "side_file": "/content/drive/MyDrive/study_knee_side.csv",
    "oai_dir": "/content/drive/MyDrive/oai",
}

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub==1.0.0", "kagglesdk==0.1.37"], check=True)
for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "KAGGLE_API_TOKEN"):
    try:
        v = userdata.get(k)
        if v:
            os.environ[k] = v
    except Exception:
        pass
import kagglehub

RUNS_DIR = Path(SETUP["runs_dir"]); RUNS_DIR.mkdir(parents=True, exist_ok=True)
DATA = Path(SETUP["data_dir"]); DATA.mkdir(parents=True, exist_ok=True)


def _get(fn, **kw):
    try:
        return fn(**kw)
    except FileExistsError:
        return fn(**kw, force_download=True)


# 1. competition CSVs (training reads the cache, not the DICOMs)
COMP_DIR = DATA / "competition"
for f in ("train.csv", "train_series.csv"):
    _get(kagglehub.competition_download, handle=SETUP["competition"], path=f, output_dir=str(COMP_DIR))

# 2. validation labels, folds, scanner metadata from Kaggle; training labels from Drive
for ref in SETUP["label_handles"]:
    _get(kagglehub.dataset_download, handle=ref, output_dir=str(DATA / "inputs" / ref.replace("/", "__")))
have = {p.name for p in (DATA / "inputs").rglob("*.csv")}
missing = sorted(set(SETUP["label_files"]) - have)
assert not missing, f"not found in label_handles: {missing}"
(DATA / "inputs" / "drive_labels").mkdir(parents=True, exist_ok=True)
for name in sorted({RUN["labels"], RUN["val_labels"]}):
    src = Path(SETUP["drive_labels"]) / f"train_labels_{name}.csv"
    assert src.exists(), f"upload {src.name} to {SETUP['drive_labels']} on Drive"
    shutil.copy(src, DATA / "inputs" / "drive_labels" / src.name)

# 3. cache shards: unpack the Drive tars to local disk (~11 GB each); finished shards are skipped
failed = []
for i in range(6):
    dest = DATA / "shards" / f"shard{i}"
    if any(dest.glob("*/preproc.json")) or any(dest.glob("*/*/preproc.json")):
        print(f"  shard {i}: already unpacked"); continue
    tar = Path(SETUP["drive_shards"]) / f"series_cache_{SETUP['cache_key']}_shard{i}.tar"
    if not tar.exists():
        failed.append(i); print(f"  shard {i}: {tar} not found"); continue
    t0 = time.time(); dest.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(["tar", "-xf", str(tar), "-C", str(dest)], capture_output=True, text=True)
    if r.returncode == 0 and (any(dest.glob("*/preproc.json")) or any(dest.glob("*/*/preproc.json"))):
        print(f"  shard {i}: unpacked in {(time.time() - t0) / 60:.1f} min | "
              f"{shutil.disk_usage(DATA).free / 1e9:.0f} GB free", flush=True)
    else:
        failed.append(i); shutil.rmtree(dest, ignore_errors=True)
        print(f"  shard {i}: FAILED {r.stderr[-300:]}", flush=True)
assert not failed, f"shards failed: {failed}. Re-run this cell; finished shards are skipped."
INPUT_ROOTS = [DATA]
print("environment ready | runs:", RUNS_DIR)


In [ ]:
# ======================= Config: CFG from the arch defaults + tuned params + RUN; derived run name =======================
import random, math, hashlib, copy
from dataclasses import dataclass
from typing import Callable
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]

ARCHS = {
    # ConvNeXt has no BatchNorm, so the whole network trains
    "convnext_tiny": {"backbone": "convnext_tiny.fb_in22k_ft_in1k", "img": 256, "unfreeze_last": 4,
                      "bn_mode": "train", "drop_path": 0.1, "lr_backbone": 1.0e-4, "lr_head": 1.0e-3,
                      "weight_decay": 0.05},
    # CoAtNet's conv stages use BatchNorm: keep pretrained statistics, train the last 2 stages
    "coatnet2":      {"backbone": "coatnet_rmlp_2_rw_224.sw_in12k_ft_in1k", "img": 224, "unfreeze_last": 2,
                      "bn_mode": "frozen", "drop_path": 0.1, "lr_backbone": 3.0e-5, "lr_head": 1.0e-3,
                      "weight_decay": 0.05},
}
ARCH = RUN["arch"]
assert ARCH in ARCHS, ARCH
assert RUN["layout"] in ("ontology5", "dense5"), RUN["layout"]
assert RUN["overlap_stride"] in (None, 1, 2), RUN["overlap_stride"]
assert not RUN["overlap_stride"] or RUN["layout"] == "dense5", "overlap windows are defined on dense5"
assert RUN["oai"] in (None, "A", "B"), RUN["oai"]

CFG = {
    "arch": ARCH,
    "seed": 42,
    "fold": RUN["folds"][0],
    "folds": list(RUN["folds"]),
    "n_folds": 5,
    "cache_key": SETUP["cache_key"],
    "layout": RUN["layout"],
    "overlap_stride": RUN["overlap_stride"],
    "laterality": "flip_left_to_right" if RUN["laterality"] else False,
    "oai": RUN["oai"],
    "oai_weight": RUN["oai_weight"] if RUN["oai"] else None,
    "labels_file": f"train_labels_{RUN['labels']}.csv",
    "val_labels_file": f"train_labels_{RUN['val_labels']}.csv",
    "gold_weight": 1.0,
    "folds_file": "folds_v31_scanner.csv",
    "meta_file": "dicom_study_metadata_train.csv",
    **ARCHS[ARCH],
    "aug_scale": 1.0,
    "study_layers": 2,
    "study_heads": 8,
    "study_dropout": 0.1,
    "grad_ckpt": False,
    "slot_prior": None,
    "epochs": RUN["epochs"],
    "ema_decay": 0.998,
    "batch": 2,
    "accum": 4,
    "num_workers": min(8, os.cpu_count() or 2),
    "time_budget_h": 11.0,
}

TUNED_FILE = RUNS_DIR / f"{ARCH}_best_params.json"
TUNED = TUNED_FILE.exists()
if TUNED:
    tuned = json.loads(TUNED_FILE.read_text())["params"]
    CFG.update(tuned)
    print("applied tuned params:", tuned)
else:
    print(f"no {TUNED_FILE.name}: arch defaults")
if RUN["img"]:
    CFG["img"] = RUN["img"]
CFG["epochs"] = RUN["epochs"]
CFG["patience"] = RUN["patience"]
CFG["save_every_epoch"] = RUN["save_every_epoch"]
CFG["select_label"] = RUN["select_label"]
CFG["smoke"] = RUN["smoke"]


def run_name(cfg, tuned, tag=""):
    """One name per distinct set-up (matches the names of the earlier runs where the set-up is the same)."""
    n = f"{cfg['arch']}_{cfg['labels_file'][len('train_labels_'):-4]}" + ("_tuned" if tuned else "")
    if cfg["layout"] != "ontology5":
        n += f"_{cfg['layout']}"
    if cfg.get("overlap_stride"):
        n += f"ov{cfg['overlap_stride']}"
    if cfg.get("laterality"):
        n += "_lat"
    if cfg.get("oai"):
        n += f"_oai{cfg['oai']}" + ("" if cfg["oai_weight"] == 1.0 else f"_w{cfg['oai_weight']:g}")
    if cfg["img"] != ARCHS[cfg["arch"]]["img"]:
        n += f"_{cfg['img']}px"
    if cfg["val_labels_file"] != cfg["labels_file"]:
        n += f"_val{cfg['val_labels_file'][len('train_labels_'):-4]}"
    return n + (f"_{tag}" if tag else "")


CFG["name"] = run_name(CFG, TUNED, RUN["tag"])

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(CFG["seed"])
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = (torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
             else torch.float16)                      # bf16 on A100/L4; fp16 + GradScaler on a T4
print("device:", DEV, torch.cuda.get_device_name(0) if DEV.type == "cuda" else "", "| precision:", AMP_DTYPE)
print("run name:", CFG["name"], "|", CFG["backbone"], f"{CFG['img']} px")


In [ ]:
# ======================= Series cache: manifests -> SERIES_PATH, PREPROC =======================
def find_dirs(prefix, bases=None):
    """Folders whose name starts with `prefix`, without crawling the DICOM folders."""
    hits = []
    for root, dirs, _ in (w for b in (bases or INPUT_ROOTS) for w in os.walk(b)):
        dirs[:] = [d for d in dirs if d not in ("train_series", "test_series")]
        for d in list(dirs):
            if d.startswith(prefix):
                hits.append(Path(root) / d)
                dirs.remove(d)
    return sorted(hits)


KEY = CFG["cache_key"]
shard_dirs = find_dirs(f"series_cache_{KEY}_shard")
manifests = []
for d in shard_dirs:
    pre = json.loads((d / "preproc.json").read_text())
    assert pre["cache_key"] == KEY, f"{d} was built with different settings: {pre['cache_key']}"
    m = pd.read_csv(d / "manifest.csv")
    m["dir"] = str(d)
    manifests.append(m)
man = pd.concat(manifests, ignore_index=True)
ok = man[man["status"] == "ok"].copy()
PREPROC = {k: pre[k] for k in pre if k != "cache_key"}
assert len(shard_dirs) == 6, f"expected 6 shards, found {len(shard_dirs)}"
assert man["SeriesInstanceUID"].duplicated().sum() == 0, "a series appears in two shards"
SERIES_PATH = {u: Path(d) / f"{u}.npz" for u, d in zip(ok["SeriesInstanceUID"], ok["dir"])}
N_CACHE = PREPROC["n_cache"]
CACHE_HW = PREPROC["img"]
print(f"{len(shard_dirs)} shards | {len(man):,} series, {len(ok):,} ok | n_cache {N_CACHE}, {CACHE_HW} px")


In [ ]:
# ======================= Layout: slots, which cached slices each slot takes, series plan =======================
SLICE_BUDGET = {"11S": 18, "11C": 15, "11A": 15, "00S": 12, "00C": 6}


@dataclass(frozen=True)
class Slot:
    name: str           # series code: fluid, fat-sat, plane (e.g. "11S")
    n_slices: int       # multiple of 3 (triplets)
    prefer: tuple       # series codes in preference order


def assign_by_preference(g, slots, tiebreak="n_files"):
    """For each slot in order: first unused series matching its preferred codes."""
    used, out = set(), []
    for s in slots:
        uid = None
        for code in s.prefer:
            c = g[(g["code"] == code) & ~g["SeriesInstanceUID"].isin(used)]
            if len(c):
                if tiebreak in c:
                    c = c.sort_values(tiebreak, ascending=False)
                uid = c["SeriesInstanceUID"].iloc[0]
                break
        used.add(uid)
        out.append(uid)
    return out


@dataclass
class Layout:
    name: str
    slots: list
    assign: Callable = None

    def __post_init__(self):
        if self.assign is None:
            self.assign = lambda g: assign_by_preference(g, self.slots)

    @property
    def slot_sizes(self):
        return [s.n_slices for s in self.slots]


def slot_take(n_slices, n_cache, overlap_stride=None):
    """Indices into a series' n_cache cached slices. Plain: n_slices evenly spaced. Overlap: 3-slice windows
    centred on 1, 1+stride, ... (stride 2: (0,1,2), (2,3,4), ...). Same rule as the submission leg's run_layout."""
    if overlap_stride:
        centres = np.arange(1, n_cache - 1, int(overlap_stride))
        t = np.concatenate([[c - 1, c, c + 1] for c in centres]).astype(int)
    else:
        t = np.linspace(0, n_cache - 1, n_slices).round().astype(int)
    assert len(t) == n_slices and n_slices % 3 == 0, (len(t), n_slices)
    return t


def make_layout(name, n_cache, overlap_stride=None):
    if name == "ontology5":
        return Layout("ontology5", [Slot(c, n, (c,)) for c, n in SLICE_BUDGET.items()])
    assert name == "dense5", name
    if overlap_stride:
        n = 3 * len(np.arange(1, n_cache - 1, int(overlap_stride)))
        return Layout(f"dense5ov{overlap_stride}", [Slot(c, n, (c,)) for c in SLICE_BUDGET])
    n = n_cache // 3 * 3
    return Layout("dense5", [Slot(c, n, (c,)) for c in SLICE_BUDGET])


LAYOUT = make_layout(CFG["layout"], N_CACHE, CFG["overlap_stride"])
TAKE = [slot_take(s.n_slices, N_CACHE, CFG["overlap_stride"]) for s in LAYOUT.slots]
N_SLOTS = len(LAYOUT.slots)
SLOT_SIZES = LAYOUT.slot_sizes
SLOT_BOUNDS = np.cumsum([0] + SLOT_SIZES)
SLOT_PLANES = [s.name[-1] for s in LAYOUT.slots]
TOTAL = sum(SLOT_SIZES)
N_TRIPLETS = TOTAL // 3
TRIPLET_SLOT = torch.tensor(np.repeat(np.arange(N_SLOTS), [n // 3 for n in SLOT_SIZES]))
assert set(SLOT_PLANES) <= {"S", "C", "A"}

ROOT = COMP_DIR
series = pd.read_csv(ROOT / "train_series.csv")
series["code"] = (series["Fluid_Sensitive"].astype(int).astype(str)
                  + series["Fat_Suppression"].astype(int).astype(str)
                  + series["Anatomical_Plane"].str[0])
series = series[series["SeriesInstanceUID"].isin(SERIES_PATH)]
series = series.merge(ok[["SeriesInstanceUID", "n_files"]], on="SeriesInstanceUID")

PLAN = {st: LAYOUT.assign(g) for st, g in series.groupby("StudyInstanceUID")}
fill = np.mean([[u is not None for u in v] for v in PLAN.values()], axis=0)
print(f"layout {LAYOUT.name}: {N_TRIPLETS} triplets per study | slices per slot {SLOT_SIZES} | first takes {TAKE[0][:9].tolist()}")
print("slot fill:", {s.name: f"{f:.1%}" for s, f in zip(LAYOUT.slots, fill)},
      f"| {len(PLAN):,} studies planned, {sum(all(u is None for u in v) for v in PLAN.values())} empty")


In [ ]:
# ======================= Labels, weights, validation reference =======================
def find_file(name, bases=None):
    for base in bases or INPUT_ROOTS:
        for root, dirs, files in os.walk(base):
            dirs[:] = [d for d in dirs if d not in ("train_series", "test_series", "shards")
                       and not d.startswith("series_cache_")]
            if name in files:
                return Path(root) / name
    raise FileNotFoundError(f"{name} not found")


lab = pd.read_csv(find_file(CFG["labels_file"]))
assert not [c for c in ["StudyInstanceUID"] + TARGETS if c not in lab.columns], "label file is missing columns"
assert lab[TARGETS].stack().between(0, 1).all(), "labels outside 0-1"

train = pd.read_csv(ROOT / "train.csv")
gold_mask = train[TARGETS].notna().all(axis=1)
GOLD_UIDS = set(train.loc[gold_mask, "StudyInstanceUID"])

extra = [c for c in lab.columns if c.endswith("__conf")]
df = train[["StudyInstanceUID"]].merge(lab[["StudyInstanceUID"] + TARGETS + extra], on="StudyInstanceUID", how="left")
g = train.loc[gold_mask].set_index("StudyInstanceUID")[TARGETS]          # the 58 gold studies keep their real labels
df = df.set_index("StudyInstanceUID")
df.loc[g.index, TARGETS] = g.values
df = df.reset_index()
df = df[df["StudyInstanceUID"].map(lambda s: s in PLAN and any(u is not None for u in PLAN[s]))].reset_index(drop=True)
df["is_gold"] = df["StudyInstanceUID"].isin(GOLD_UIDS)


def label_weights(frame, gold_uids, gold_weight):
    y = frame[TARGETS].to_numpy(np.float32)
    known = ~np.isnan(y)
    conf_cols = [f"{t}__conf" for t in TARGETS]
    if all(c in frame.columns for c in conf_cols):
        w = 0.25 + 0.75 * frame[conf_cols].fillna(0).to_numpy(np.float32)
    else:
        w = np.ones_like(y)
    w = np.where(known, w, 0.0)
    gold = frame["StudyInstanceUID"].isin(gold_uids).to_numpy()
    w[gold] = np.where(known[gold], gold_weight, 0.0)
    return np.nan_to_num(y, nan=0.0), w.astype(np.float32)


Y, W = label_weights(df, GOLD_UIDS, CFG["gold_weight"])
has_label = W.sum(axis=1) > 0
df = df[has_label].reset_index(drop=True)
Y, W = Y[has_label], W[has_label]

ref = pd.read_csv(find_file(CFG["val_labels_file"])).set_index("StudyInstanceUID")[TARGETS]
Y_REF = ref.reindex(df["StudyInstanceUID"]).to_numpy(np.float32)
gold_rows = df["is_gold"].to_numpy()
Y_REF[gold_rows] = Y[gold_rows]
assert not np.isnan(Y_REF).any(), "reference labels missing for some training studies"
print(f"{len(df):,} RSNA studies ({df['is_gold'].sum()} gold) | dropped {(~has_label).sum()} without labels | "
      f"train {CFG['labels_file']} | validate on {CFG['val_labels_file']}")


In [ ]:
# ======================= Folds (fixed file; scanner groups) =======================
meta = pd.read_csv(find_file(CFG["meta_file"]))
SCANNER_COLS = ["Manufacturer", "ManufacturerModelName", "MagneticFieldStrength"]
fp = meta.set_index("StudyInstanceUID")[SCANNER_COLS].astype(str).agg("|".join, axis=1)
df["scanner"] = df["StudyInstanceUID"].map(fp)
no_meta = df["scanner"].isna() | df["scanner"].str.contains(r"^nan\|nan\|nan$")
df.loc[no_meta, "scanner"] = "unknown_" + df.loc[no_meta, "StudyInstanceUID"]
df["group"] = df["scanner"]

ff = pd.read_csv(find_file(CFG["folds_file"]))
known = dict(zip(ff["StudyInstanceUID"], ff["fold"]))
g2f = ff.groupby("group")["fold"].agg(lambda s: int(s.mode().iloc[0])).to_dict()
df["fold"] = df["StudyInstanceUID"].map(known)
new = df["fold"].isna()
df.loc[new, "fold"] = df.loc[new, "group"].map(g2f)
left = df["fold"].isna()
df.loc[left, "fold"] = [int(hashlib.md5(u.encode()).hexdigest(), 16) % CFG["n_folds"]
                        for u in df.loc[left, "StudyInstanceUID"]]
df["fold"] = df["fold"].astype(int)
assert (df["fold"] >= 0).all()
print(df.groupby("fold").agg(studies=("StudyInstanceUID", "size"), gold=("is_gold", "sum")).T.to_string())


In [ ]:
# ======================= Knee side (laterality) =======================
KNEE_SIDE = {}
if CFG["laterality"]:
    _side_df = pd.read_csv(SETUP["side_file"], keep_default_na=False)
    KNEE_SIDE = dict(zip(_side_df["StudyInstanceUID"], _side_df["side"]))
    print("knee side (RSNA):", pd.Series([KNEE_SIDE.get(u, "?") for u in df["StudyInstanceUID"]]).value_counts().to_dict())
else:
    print("laterality off")


In [ ]:
# ======================= OAI knees as extra TRAINING studies (fold -1, never validation) =======================
N_OAI = 0
if CFG["oai"]:
    from concurrent.futures import ThreadPoolExecutor
    OAI_DRIVE = Path(SETUP["oai_dir"])
    OAI_SHARD_DRIVE = OAI_DRIVE / f"series_cache_{KEY}_shard_oai"
    OAI_SHARD_LOCAL = Path("/content/oai_cache")       # reading 9.6k npz from Drive every epoch is slow
    OAI_LABELS = (["PF OA", "Lateral OA", "Synovitis"] if CFG["oai"] == "A" else
                  ["ACL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA", "PF OA",
                   "Effusion", "Synovitis", "Baker's"])
    assert not df["StudyInstanceUID"].astype(str).str.startswith("OAI_").any(), "OAI rows already present: re-run from the labels cell"

    # same preprocessing as the RSNA cache
    _oai_pre = json.loads((OAI_SHARD_DRIVE / "preproc.json").read_text())
    for k in ["n_cache", "span", "img", "pct", "crop_mm"]:
        assert _oai_pre.get(k) == PREPROC.get(k), f"OAI cache preproc[{k}] {_oai_pre.get(k)} != {PREPROC.get(k)}"

    # local copy (16 threads; Drive is slow per file)
    _man = pd.read_csv(OAI_SHARD_DRIVE / "manifest.csv")
    _man = _man[_man["status"] == "ok"].drop_duplicates("SeriesInstanceUID")
    OAI_SHARD_LOCAL.mkdir(parents=True, exist_ok=True)
    _t = time.time()
    _have = {p.name for p in OAI_SHARD_LOCAL.glob("*.npz")}
    _todo = [u for u in _man["SeriesInstanceUID"] if f"{u}.npz" not in _have]

    def _cp(u):
        shutil.copy2(OAI_SHARD_DRIVE / f"{u}.npz", OAI_SHARD_LOCAL / f"{u}.npz")

    with ThreadPoolExecutor(16) as _ex:
        for i, _ in enumerate(_ex.map(_cp, _todo)):
            if (i + 1) % 1000 == 0:
                print(f"  copied {i + 1:,}/{len(_todo):,} ({time.time() - _t:.0f}s)", flush=True)
    print(f"OAI cache: {len(_man):,} series local ({time.time() - _t:.0f}s)")

    # series table + plan (same layout.assign as RSNA)
    _ser = _man[["SeriesInstanceUID", "n_files"]].copy()
    _ser["StudyInstanceUID"] = _ser["SeriesInstanceUID"].str.rsplit("_", n=1).str[0]
    _ser["code"] = _ser["SeriesInstanceUID"].str.rsplit("_", n=1).str[1]
    for u in _ser["SeriesInstanceUID"]:
        SERIES_PATH[u] = OAI_SHARD_LOCAL / f"{u}.npz"
    _oai_plan = {st: LAYOUT.assign(g) for st, g in _ser.groupby("StudyInstanceUID")}
    PLAN.update(_oai_plan)

    # labels -> Y / W rows (weight 0 where OAI has no label or the label is not in this run)
    _lab = pd.read_csv(OAI_DRIVE / "oai_v00_moaks_soft_labels.csv", dtype={"ID": str})
    _lab["StudyInstanceUID"] = "OAI_" + _lab["ID"] + "_" + _lab["side"]
    _lab = _lab[_lab["StudyInstanceUID"].isin(_oai_plan)].reset_index(drop=True)
    _L = _lab.reindex(columns=TARGETS).astype(float)
    for t in TARGETS:
        if t not in OAI_LABELS:
            _L[t] = np.nan
    _Y_oai = np.nan_to_num(_L.to_numpy(), nan=0.0).astype(Y.dtype)
    _W_oai = (_L.notna().to_numpy() * CFG["oai_weight"]).astype(W.dtype)

    _rows = pd.DataFrame({"StudyInstanceUID": _lab["StudyInstanceUID"], "fold": -1, "group": "oai", "is_gold": False})
    for c in df.columns:
        if c not in _rows:
            _rows[c] = np.nan if df[c].dtype.kind in "fc" else None
    df = pd.concat([df, _rows[df.columns]], ignore_index=True)
    df["is_gold"] = df["is_gold"].fillna(False).astype(bool)
    Y = np.concatenate([Y, _Y_oai])
    W = np.concatenate([W, _W_oai])
    Y_REF = np.concatenate([Y_REF, np.zeros_like(_Y_oai, dtype=Y_REF.dtype)])
    assert len(df) == len(Y) == len(W) == len(Y_REF)
    if CFG["laterality"]:
        KNEE_SIDE.update(dict(zip(_lab["StudyInstanceUID"], _lab["side"])))
    N_OAI = len(_lab)
    print(f"OAI {CFG['oai']}: {N_OAI:,} knees, weight {CFG['oai_weight']} | labelled per target:",
          {t: int(n) for t, n in zip(TARGETS, (_W_oai > 0).sum(0)) if n})
else:
    print("OAI off")


In [ ]:
# ======================= Dataset, GPU augmentation, model input =======================
from torch.utils.data import Dataset, DataLoader


class KneeDataset(Dataset):
    """One study: uint8 (TOTAL, 384, 384) slices in layout order, slot presence, 12 targets, 12 weights.
    Each slot takes TAKE[k] of its series' cached slices; left knees are then flipped to right-knee orientation
    (coronal/axial left<->right, sagittal slice order reversed) when laterality is on."""

    def __init__(self, frame, y, w):
        self.uids = frame["StudyInstanceUID"].tolist()
        self.y, self.w = y, w
        self.flip = bool(CFG["laterality"])

    def __len__(self):
        return len(self.uids)

    def __getitem__(self, i):
        chosen = PLAN[self.uids[i]]
        parts, present = [], []
        for s, uid, idx in zip(LAYOUT.slots, chosen, TAKE):
            p = SERIES_PATH.get(uid) if uid else None
            if p is not None:
                parts.append(np.load(p)["x"][idx])
                present.append(1.0)
            else:
                parts.append(np.zeros((s.n_slices, CACHE_HW, CACHE_HW), np.uint8))
                present.append(0.0)
        vol = torch.from_numpy(np.ascontiguousarray(np.concatenate(parts)))
        if self.flip and KNEE_SIDE.get(self.uids[i]) == "L":
            vol = torch.cat([torch.flip(vol[SLOT_BOUNDS[k]:SLOT_BOUNDS[k + 1]], dims=[0] if plane == "S" else [-1])
                             for k, plane in enumerate(SLOT_PLANES)]).contiguous()
        return (vol, torch.tensor(present, dtype=torch.float32),
                torch.from_numpy(self.y[i]), torch.from_numpy(self.w[i]))


def make_ds(idx):
    return KneeDataset(df.iloc[idx], Y[idx], W[idx])


AUG = {k: v * CFG.get("aug_scale", 1.0) for k, v in {"rot_deg": 8.0, "zoom": 0.08, "shift": 0.05, "intensity": 0.10}.items()}
MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEV).view(1, 1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=DEV).view(1, 1, 3, 1, 1)


def gpu_augment(x):
    """x: float (B, TOTAL, H, W) in [0,1]. One random warp per (study, slot), shared by the slot's slices."""
    out, start = [], 0
    for n in SLOT_SIZES:
        xs = x[:, start:start + n]
        B, dev = xs.shape[0], xs.device
        rot = (torch.rand(B, device=dev) * 2 - 1) * math.radians(AUG["rot_deg"])
        zoom = 1 + torch.rand(B, device=dev) * AUG["zoom"]
        tx = (torch.rand(B, device=dev) * 2 - 1) * AUG["shift"]
        ty = (torch.rand(B, device=dev) * 2 - 1) * AUG["shift"]
        cos, sin = torch.cos(rot) / zoom, torch.sin(rot) / zoom
        theta = torch.stack([torch.stack([cos, -sin, tx], 1), torch.stack([sin, cos, ty], 1)], 1)
        grid = F.affine_grid(theta, xs.shape, align_corners=False)
        xs = F.grid_sample(xs, grid, mode="bilinear", padding_mode="border", align_corners=False)
        xs = xs * (1 + (torch.rand(B, 1, 1, 1, device=dev) * 2 - 1) * AUG["intensity"])
        out.append(xs)
        start += n
    return torch.cat(out, 1).clamp(0, 1)


def to_model_input(x_uint8, train):
    """uint8 (B, TOTAL, 384, 384) -> normalised float (B, N_TRIPLETS, 3, img, img); consecutive slices -> RGB."""
    x = x_uint8.to(DEV, non_blocking=True).float().div_(255.0)
    if x.shape[-1] != CFG["img"]:
        x = F.interpolate(x, size=(CFG["img"], CFG["img"]), mode="bilinear", align_corners=False, antialias=True)
    if train:
        x = gpu_augment(x)
    B = x.shape[0]
    x = x.view(B, N_TRIPLETS, 3, CFG["img"], CFG["img"])
    return (x - MEAN) / STD


In [ ]:
# ======================= Model =======================
import timm


class TimmEncoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.m = timm.create_model(CFG["backbone"], pretrained=pretrained, num_classes=0, global_pool="avg",
                                   drop_path_rate=CFG.get("drop_path", 0.0))
        self.num_features = self.m.num_features
        if CFG.get("grad_ckpt"):
            self.m.set_grad_checkpointing(True)

    def forward(self, x):
        return self.m(x.contiguous(memory_format=torch.channels_last))


class KneeNet(nn.Module):
    """CNN per triplet -> tokens + slot/position embedding -> study transformer -> per-finding attention -> 12 logits."""

    def __init__(self, n_out, n_slots, triplet_slot, img, pretrained=True, prior=None):
        super().__init__()
        self.enc = TimmEncoder(pretrained=pretrained).to(memory_format=torch.channels_last)
        d = self.enc.num_features
        T = len(triplet_slot)
        self.register_buffer("triplet_slot", triplet_slot.clone())
        self.slot_emb = nn.Parameter(torch.zeros(n_slots, d))
        self.pos_emb = nn.Parameter(torch.zeros(T, d))
        self.norm = nn.LayerNorm(d)
        L = CFG.get("study_layers", 0)
        self.study = None
        if L:
            layer = nn.TransformerEncoderLayer(d, CFG["study_heads"], dim_feedforward=2 * d,
                                               dropout=CFG["study_dropout"], activation="gelu",
                                               batch_first=True, norm_first=True)
            self.study = nn.TransformerEncoder(layer, L, enable_nested_tensor=False)
            self.post = nn.LayerNorm(d)
        self.att = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, n_out))
        self.drop = nn.Dropout(0.2)
        self.cls_w = nn.Parameter(torch.randn(n_out, d) * 0.02)
        self.cls_b = nn.Parameter(torch.zeros(n_out))
        p = torch.zeros(n_out, n_slots) if prior is None else torch.as_tensor(prior, dtype=torch.float32)
        self.register_buffer("prior", p)
        self.frozen_bn = []

    def train(self, mode=True):
        super().train(mode)
        for m in self.frozen_bn:
            m.eval()
        return self

    def forward(self, x, present):
        B, T = x.shape[:2]
        f = self.enc(x.flatten(0, 1)).view(B, T, -1)
        f = self.norm(f + self.slot_emb[self.triplet_slot] + self.pos_emb)
        keep = present[:, self.triplet_slot] > 0
        if self.study is not None:
            f = self.post(self.study(f, src_key_padding_mask=~keep))
        a = self.att(f) + self.prior[:, self.triplet_slot].T.unsqueeze(0)
        a = a.float().masked_fill(~keep.unsqueeze(-1), -1e4).softmax(dim=1)
        pooled = torch.einsum("btn,btd->bnd", a, f.float())
        return (self.drop(pooled) * self.cls_w).sum(-1) + self.cls_b


def freeze_backbone(model, unfreeze_last):
    m = model.enc.m
    for p in m.parameters():
        p.requires_grad = False
    stages = list(m.stages)
    trainable = stages[len(stages) - unfreeze_last:]
    if unfreeze_last >= len(stages):
        trainable.append(m.stem)
    in_body = {id(p) for mod in [m.stem, *stages] for p in mod.parameters()}
    for mod in trainable:
        for p in mod.parameters():
            p.requires_grad = True
    for p in m.parameters():
        if id(p) not in in_body:
            p.requires_grad = True
    model.frozen_bn = [mod for mod in m.modules() if isinstance(mod, nn.BatchNorm2d)
                       and (CFG.get("bn_mode") == "frozen" or not any(p.requires_grad for p in mod.parameters()))]
    if CFG.get("bn_mode") == "frozen":
        for mod in model.frozen_bn:
            for p in mod.parameters():
                p.requires_grad = False
    model.train(model.training)


def build_optimizer(model):
    enc = [p for p in model.enc.parameters() if p.requires_grad]
    head = [p for n, p in model.named_parameters() if not n.startswith("enc.") and p.requires_grad]
    return torch.optim.AdamW([{"params": enc, "lr": CFG["lr_backbone"]}, {"params": head, "lr": CFG["lr_head"]}],
                             weight_decay=CFG["weight_decay"])


class EMA:
    """EMA of all weights and buffers; the EMA copy is validated and saved."""

    def __init__(self, model, decay):
        self.module = copy.deepcopy(model).eval()
        for p in self.module.parameters():
            p.requires_grad_(False)
        self.decay, self.updates = decay, 0

    @torch.no_grad()
    def update(self, model):
        self.updates += 1
        d = min(self.decay, (1 + self.updates) / (10 + self.updates))
        msd = model.state_dict()
        for k, v in self.module.state_dict().items():
            if v.dtype.is_floating_point:
                v.lerp_(msd[k].detach(), 1 - d)
            else:
                v.copy_(msd[k])


In [ ]:
# ======================= CHECK: full set-up, consistency, one batch, one training step =======================
import matplotlib.pyplot as plt

n_oai_rows = int(df["StudyInstanceUID"].astype(str).str.startswith("OAI_").sum())
print("run name     :", CFG["name"])
print("backbone     :", CFG["backbone"], f"{CFG['img']} px | tuned params: {TUNED} | epochs {CFG['epochs']} | patience {CFG['patience']}")
print("layout       :", LAYOUT.name, "| triplets", N_TRIPLETS, "| slices per slot", SLOT_SIZES)
print("laterality   :", CFG["laterality"], "| knee side known for",
      f"{np.mean([KNEE_SIDE.get(u, '?') in ('L', 'R') for u in df['StudyInstanceUID']]):.1%} of rows" if CFG["laterality"] else "")
print("OAI          :", CFG["oai"], f"| {n_oai_rows:,} rows | weight {CFG['oai_weight']}")
print("labels       :", CFG["labels_file"], "-> validate on", CFG["val_labels_file"])
print("folds        :", CFG["folds"], "| select_label", CFG["select_label"], "| save_every_epoch", CFG["save_every_epoch"],
      "| smoke", CFG["smoke"])

# consistency
expected_tr = {("ontology5", None): 22, ("dense5", None): 40, ("dense5", 2): 55, ("dense5", 1): 110}[(CFG["layout"], CFG["overlap_stride"])]
assert N_CACHE == 24 and N_TRIPLETS == expected_tr, (N_CACHE, N_TRIPLETS, expected_tr)
assert n_oai_rows == N_OAI and (n_oai_rows > 2000) == bool(CFG["oai"]), (n_oai_rows, CFG["oai"])
assert all(np.array_equal(t, slot_take(s.n_slices, N_CACHE, CFG["overlap_stride"])) for t, s in zip(TAKE, LAYOUT.slots))
assert CFG["name"] == run_name(CFG, TUNED, RUN["tag"]), "settings changed after the config cell: re-run from there"

# existing folds under this name must have been trained with the same set-up (no name collisions)
SIG_KEYS = ["backbone", "img", "layout", "overlap_stride", "laterality", "oai", "oai_weight", "labels_file", "epochs"]
for f in range(CFG["n_folds"]):
    rc = RUNS_DIR / f"{CFG['name']}_fold{f}" / "run_config.json"
    if rc.exists():
        old = json.loads(rc.read_text())
        diff = {k: (old.get(k), CFG.get(k)) for k in SIG_KEYS if k in old and old.get(k) != CFG.get(k)}
        done = (rc.parent / "done.json").exists()
        print(f"  fold {f}: existing folder ({'done, will be skipped' if done else 'unfinished, will resume'})"
              + (f"  DIFFERENT SETTINGS {diff}" if diff else ""))
        assert not diff, f"fold {f} folder was trained with other settings: change RUN['tag']"

# one batch through the real path (fold of CFG['folds'][0]); visual laterality check on 11C
_va = np.flatnonzero(df["fold"].to_numpy() == CFG["folds"][0])
_ds = make_ds(_va)
t0 = time.time()
_b = [_ds[j] for j in range(4)]
xb = torch.stack([b[0] for b in _b]); pb = torch.stack([b[1] for b in _b])
print(f"4 studies loaded in {time.time() - t0:.2f}s | raw {tuple(xb.shape)} | model input {tuple(to_model_input(xb[:2], train=True).shape)}")
if CFG["laterality"]:
    _ci = [s.name for s in LAYOUT.slots].index("11C")
    _pick = {"R": [], "L": []}
    for _j, _u in enumerate(_ds.uids):
        _sd = KNEE_SIDE.get(_u)
        if _sd in _pick and len(_pick[_sd]) < 3 and PLAN[_u][_ci] is not None:
            _pick[_sd].append(_j)
        if all(len(v) == 3 for v in _pick.values()):
            break
    fig, axes = plt.subplots(2, 3, figsize=(10, 7))
    for r, sd in enumerate(["R", "L"]):
        for c, j in enumerate(_pick[sd]):
            v = _ds[j][0]
            axes[r, c].imshow(v[SLOT_BOUNDS[_ci] + SLOT_SIZES[_ci] // 2], cmap="gray"); axes[r, c].axis("off")
            axes[r, c].set_title(f"{sd} knee" + (" (after flip)" if sd == "L" else ""), fontsize=9)
    plt.suptitle("11C middle slice: medial compartment on the SAME side in every image", fontsize=10)
    plt.show()

# one training step: memory and speed
torch.cuda.reset_peak_memory_stats()
_m = KneeNet(len(TARGETS), N_SLOTS, TRIPLET_SLOT, CFG["img"], pretrained=True, prior=CFG["slot_prior"]).to(DEV)
freeze_backbone(_m, CFG["unfreeze_last"])
_opt = build_optimizer(_m)
_m.train(); t0 = time.time()
with torch.autocast("cuda", dtype=AMP_DTYPE):
    _lg = _m(to_model_input(xb[:CFG["batch"]], train=True), pb[:CFG["batch"]].to(DEV))
_lg.float().mean().backward(); torch.cuda.synchronize()
print(f"one step: logits {tuple(_lg.shape)} | {time.time() - t0:.2f}s | peak GPU {torch.cuda.max_memory_allocated() / 1024**3:.1f} GB")
del _m, _opt, _lg
torch.cuda.empty_cache()
print("CHECK OK")


In [ ]:
# ======================= Training (resumable per epoch; finished folds skipped) =======================
from sklearn.metrics import roc_auc_score
from torch.optim.lr_scheduler import LambdaLR


def aucs_by_label(y_true, y_pred):
    out = {}
    for j, t in enumerate(TARGETS):
        yt = y_true[:, j] > 0.5
        out[t] = roc_auc_score(yt, y_pred[:, j]) if 0 < yt.sum() < len(yt) else np.nan
    return out


@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = []
    for xb, pb, _, _ in loader:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(to_model_input(xb, train=False), pb.to(DEV))
        preds.append(torch.sigmoid(logits.float()).cpu().numpy())
    pv = np.concatenate(preds)
    bad = ~np.isfinite(pv).all(axis=1)
    if bad.any():
        print(f"  WARNING: {bad.sum()} of {len(pv)} validation studies gave non-finite predictions (set to 0.5)", flush=True)
        pv[~np.isfinite(pv)] = 0.5
    return pv


def _oof_frame(pv, va_idx, fold, gold_val):
    o = pd.DataFrame(pv, columns=TARGETS)
    o.insert(0, "StudyInstanceUID", df["StudyInstanceUID"].to_numpy()[va_idx])
    o.insert(1, "fold", fold)
    o["is_gold"] = gold_val
    return o


def train_one_fold(fold):
    """Train one fold into RUNS_DIR/<name>_fold<k>/ (model.pt, oof.csv, folds.csv, run_config.json)."""
    out = RUNS_DIR / f"{CFG['name']}_fold{fold}"
    if (out / "done.json").exists() and not CFG["smoke"]:
        print(f"fold {fold}: already done, skipping ({out})")
        return json.loads((out / "done.json").read_text())
    out.mkdir(parents=True, exist_ok=True)
    CFG["fold"] = fold
    (out / "run_config.json").write_text(json.dumps(CFG, indent=2, default=str))
    df[["StudyInstanceUID", "fold", "group"]].to_csv(out / "folds.csv", index=False)

    tr_idx = np.flatnonzero(df["fold"].to_numpy() != fold)
    va_idx = np.flatnonzero(df["fold"].to_numpy() == fold)
    epochs = CFG["epochs"]
    if CFG["smoke"]:
        tr_idx, va_idx, epochs = tr_idx[:200], va_idx[:100], 1
    train_dl = DataLoader(make_ds(tr_idx), batch_size=CFG["batch"], shuffle=True, drop_last=True,
                          num_workers=CFG["num_workers"], pin_memory=True, persistent_workers=True)
    valid_dl = DataLoader(make_ds(va_idx), batch_size=CFG["batch"] * 2, shuffle=False,
                          num_workers=CFG["num_workers"], pin_memory=True, persistent_workers=True)
    print(f"\n===== fold {fold}: {len(tr_idx):,} train / {len(va_idx):,} valid -> {out}", flush=True)

    seed_everything(CFG["seed"])
    model = KneeNet(len(TARGETS), N_SLOTS, TRIPLET_SLOT, CFG["img"], pretrained=True, prior=CFG["slot_prior"]).to(DEV)
    freeze_backbone(model, CFG["unfreeze_last"])
    opt = build_optimizer(model)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP_DTYPE == torch.float16)
    trainable = [p for p in model.parameters() if p.requires_grad]
    ema = EMA(model, CFG["ema_decay"])

    steps_per_epoch = len(train_dl) // CFG["accum"]
    total_steps = steps_per_epoch * epochs
    warm = max(1, int(0.05 * total_steps))
    sched = LambdaLR(opt, lambda s: (s + 1) / warm if s < warm
                     else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total_steps - warm))))

    y_val = Y_REF[va_idx]
    gold_val = df["is_gold"].to_numpy()[va_idx]
    best_auc, history, t_run = -1.0, [], time.time()

    # resume (model/optimizer/scheduler/EMA saved after every epoch); only under identical settings
    ckpt_path, start_ep = out / "resume.pt", 0
    run_sig = {k: CFG.get(k) for k in ("backbone", "img", "unfreeze_last", "bn_mode", "drop_path", "lr_backbone", "lr_head",
                                       "weight_decay", "epochs", "accum", "labels_file", "cache_key", "study_layers",
                                       "ema_decay", "aug_scale", "layout", "overlap_stride", "laterality", "oai", "oai_weight")}
    run_sig["amp"] = str(AMP_DTYPE)
    if ckpt_path.exists() and not CFG["smoke"]:
        ck = torch.load(ckpt_path, map_location=DEV, weights_only=False)
        if ck.get("sig") != run_sig:
            os.replace(ckpt_path, out / f"resume_stale_{int(time.time())}.pt")
            print("  resume.pt was saved under different settings; starting fresh", flush=True)
    if ckpt_path.exists() and not CFG["smoke"]:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        scaler.load_state_dict(ck["scaler"]); sched.load_state_dict(ck["sched"])
        ema.module.load_state_dict(ck["ema"]); ema.updates = ck["ema_updates"]
        start_ep, best_auc, history = ck["epoch"] + 1, ck["best_auc"], ck["history"]
        torch.manual_seed(CFG["seed"] + start_ep)
        print(f"  resuming fold {fold} at epoch {start_ep} (best so far {best_auc:.4f})", flush=True)

    best_ep = max([h["epoch"] for h in history if h["val_auc"] >= best_auc], default=start_ep - 1)
    sel_label = CFG.get("select_label")

    for ep in range(start_ep, epochs):
        model.train()
        t0, run_loss, n_bad = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for i, (xb, pb, yb, wb) in enumerate(train_dl):
            with torch.autocast("cuda", dtype=AMP_DTYPE):
                logits = model(to_model_input(xb, train=True), pb.to(DEV))
            wb, yb = wb.to(DEV), yb.to(DEV)
            loss = (F.binary_cross_entropy_with_logits(logits.float(), yb, reduction="none") * wb).sum() / wb.sum().clamp(min=1)
            if torch.isfinite(loss):
                scaler.scale(loss / CFG["accum"]).backward()
                run_loss += loss.item()
            else:
                n_bad += 1
                if n_bad <= 3:
                    print(f"  non-finite loss at step {i}; skipped", flush=True)
            if (i + 1) % CFG["accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(trainable, 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
                sched.step()
                ema.update(model)
            if ep == start_ep and i == 30:
                per_batch = (time.time() - t0) / 31
                print(f"  ~{per_batch * len(train_dl) / 60:.0f} min per epoch, "
                      f"~{per_batch * len(train_dl) * (epochs - start_ep) / 3600:.1f} h for the remaining epochs", flush=True)

        pv = predict(ema.module, valid_dl)
        per = aucs_by_label(y_val, pv)
        macro_auc = float(np.nanmean(list(per.values())))
        auc = float(per[sel_label]) if sel_label else macro_auc
        g = aucs_by_label(y_val[gold_val], pv[gold_val]) if gold_val.sum() else {}
        g_auc = float(np.nanmean(list(g.values()))) if g else np.nan
        n_ok = max(1, len(train_dl) - n_bad)
        history.append({"epoch": ep, "loss": run_loss / n_ok, "skipped": n_bad, "val_auc": auc, "macro_auc": macro_auc, "gold_auc": g_auc})
        sel_txt = f"{sel_label} AUC {auc:.4f} (macro {macro_auc:.4f})" if sel_label else f"val AUC {auc:.4f}"
        print(f"epoch {ep}: loss {run_loss/n_ok:.4f}  skipped {n_bad}  {sel_txt}  "
              f"gold AUC {g_auc:.3f} (n={gold_val.sum()})  {(time.time()-t0)/60:.1f} min", flush=True)

        if CFG.get("save_every_epoch"):
            _oof_frame(pv, va_idx, fold, gold_val).to_csv(out / f"oof_ep{ep:02d}.csv", index=False)

        if auc > best_auc:
            best_auc, best_ep = auc, ep
            torch.save({"model": ema.module.state_dict(), "ema": True, "cfg": CFG, "targets": TARGETS,
                        "slots": [s.name for s in LAYOUT.slots], "slot_sizes": SLOT_SIZES,
                        "preproc": PREPROC, "cache_key": KEY, "epoch": ep,
                        "val_auc": auc, "macro_auc": macro_auc, "select_label": sel_label, "val_per_label": per}, out / "model.pt")
            _oof_frame(pv, va_idx, fold, gold_val).to_csv(out / "oof.csv", index=False)
            print(f"  saved best (epoch {ep})")

        if not CFG["smoke"]:
            torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                        "sched": sched.state_dict(), "epoch": ep, "best_auc": best_auc, "history": history, "sig": run_sig,
                        "ema": ema.module.state_dict(), "ema_updates": ema.updates}, out / "resume.tmp")
            os.replace(out / "resume.tmp", ckpt_path)

        if CFG.get("patience") and ep - best_ep >= CFG["patience"]:
            print(f"  early stop: no gain for {CFG['patience']} epochs (best {best_auc:.4f} at epoch {best_ep})", flush=True)
            break
        if (time.time() - t_run) / 3600 > CFG["time_budget_h"]:
            print("time budget reached; stopping")
            break

    results = {"fold": fold, "best_val_auc": best_auc, "history": history,
               "best_per_label": torch.load(out / "model.pt", weights_only=False)["val_per_label"]}
    (out / "run_config.json").write_text(json.dumps({**CFG, "results": results}, indent=2, default=float))
    if not CFG["smoke"]:
        (out / "done.json").write_text(json.dumps(results, indent=2, default=float))
        ckpt_path.unlink(missing_ok=True)
    print(f"\nfold {fold} best per-label AUC:\n" + pd.Series(results["best_per_label"]).round(3).to_string())
    del model, ema, opt, scaler, train_dl, valid_dl
    torch.cuda.empty_cache()
    return results


ALL_RESULTS = {f: train_one_fold(f) for f in CFG["folds"]}
print("\n" + pd.Series({f: r["best_val_auc"] for f, r in ALL_RESULTS.items()}, name="best val AUC").round(4).to_string())


In [ ]:
# ======================= Upload finished folds to Kaggle (trained weights only; never OAI data) =======================
UPLOAD = False
UPLOAD_HANDLE = f"harrisonhoran/{CFG['name'].replace('_', '-').replace('.', '-')}-folds"

if UPLOAD:
    stage = Path("/content/upload") / CFG["name"]
    shutil.rmtree(stage, ignore_errors=True)
    for f in range(CFG["n_folds"]):
        src_dir = RUNS_DIR / f"{CFG['name']}_fold{f}"
        if (src_dir / "done.json").exists():
            dst = stage / f"fold{f}"; dst.mkdir(parents=True)
            for name in ("model.pt", "oof.csv", "run_config.json"):       # folds.csv lists OAI study IDs: not uploaded
                shutil.copy(src_dir / name, dst / name)
    done = sorted(p.name for p in stage.iterdir())
    print("uploading", done, "->", UPLOAD_HANDLE)
    kagglehub.dataset_upload(UPLOAD_HANDLE, str(stage), version_notes=f"{CFG['name']} {done}")
else:
    print("upload off | would upload to", UPLOAD_HANDLE)
